In [ ]:
options_4 = pd.read_excel("4_days_AMZ.xlsx")
options_25 = pd.read_excel("25_days_AMZ.xlsx")
options_46 = pd.read_excel("46_days_AMZ.xlsx")
options_102 = pd.read_excel("102_days_AMZ.xlsx")

In [ ]:
def transform_df(df, maturity):
    df = df.copy()
    call_cols = df.columns[:6]  # Les 6 premières colonnes = Calls
    put_cols = df.columns[5:]   # Les suivantes = Puts

    # Transformation des DataFrames
    call_df = df[call_cols]
    call_df["Maturity"] = maturity

    put_df = df[put_cols]
    put_df["Maturity"] = maturity

    return call_df, put_df

# Transformation de chaque DataFrame
calls_4, puts_4 = transform_df(options_4, 4)
calls_25, puts_25 = transform_df(options_25, 25)
calls_46, puts_46 = transform_df(options_46, 46)
calls_102, puts_102 = transform_df(options_102, 102)

# Fusion des DataFrames par type
calls_df = pd.concat([calls_4, calls_25, calls_46, calls_102])
puts_df = pd.concat([puts_4, puts_25, puts_46, puts_102])

In [ ]:
def HestonCFOld(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, phi):
    
    xx = np.log(Spot)
    
    rvpj = rho * volvol * phi * 1j
    
    u1 = 0.5
    u2 = -0.5
    
    aa = kappa * theta
    
    b1 = kappa - rho * volvol
    b2 = kappa
    
    d1aux = np.power(rvpj - b1, 2) - np.power(volvol, 2) * (2 * u1 * phi * 1j - np.power(phi, 2))
    d2aux = np.power(rvpj - b2, 2) - np.power(volvol, 2) * (2 * u2 * phi * 1j - np.power(phi, 2))
    d1 = np.power(d1aux, 0.5)
    d2 = np.power(d2aux, 0.5)
    
    g1 = (b1 - rvpj + d1) / (b1 - rvpj - d1)
    g2 = (b2 - rvpj + d2) / (b2 - rvpj - d2)
    
    D1aux = (1 - np.exp(d1 * Maturity)) / (1 - g1 * np.exp(d1 * Maturity))
    D2aux = (1 - np.exp(d2 * Maturity)) / (1 - g2 * np.exp(d2 * Maturity))
    D1 = ((b1 - rvpj + d1) / np.power(volvol, 2)) * D1aux
    D2 = ((b2 - rvpj + d2) / np.power(volvol, 2)) * D2aux
    
    C1aux = (b1 - rvpj + d1) * Maturity - 2 * np.log((1 - g1 * np.exp(d1 * Maturity)) / (1 - g1))
    C2aux = (b2 - rvpj + d2) * Maturity - 2 * np.log((1 - g2 * np.exp(d2 * Maturity)) / (1 - g2))
    C1 = (rate - div) * phi * Maturity * 1j + (aa / np.power(volvol, 2)) * C1aux
    C2 = (rate - div) * phi * Maturity * 1j + (aa / np.power(volvol, 2)) * C2aux
    
    CF1 = np.exp(C1 + D1 * var0 + xx * phi * 1j)
    CF2 = np.exp(C2 + D2 * var0 + xx * phi * 1j)
    
    return CF1, CF2

def HestonCF(phi, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2):
    
    xx = np.log(Spot)
    
    rvpj = rho * volvol * phi * 1j
    
    uu = np.array([0.5, -0.5])[P1P2-1]
    
    aa = kappa * theta
    
    bb = np.array([kappa - rho * volvol, kappa])[P1P2-1]
    
    ddaux = np.power(rvpj - bb, 2) - np.power(volvol, 2) * (2 * uu * phi * 1j - np.power(phi, 2))

    dd = np.power(ddaux, 0.5)
    
    gg = (bb - rvpj + dd) / (bb - rvpj - dd)
    
    DDaux = (1 - np.exp(dd * Maturity)) / (1 - gg * np.exp(dd * Maturity))
    DD = ((bb - rvpj + dd) / np.power(volvol, 2)) * DDaux
    
    CCaux = (bb - rvpj + dd) * Maturity - 2 * np.log((1 - gg * np.exp(dd * Maturity)) / (1 - gg))
    CC = (rate - div) * phi * Maturity * 1j + (aa / np.power(volvol, 2)) * CCaux
    
    CF1 = np.exp(CC + DD * var0 + xx * phi * 1j)
    
    return CF1
def IntFuncHeston(phi, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike):
    
    CF1 = HestonCF(phi, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2)
    
    Output = np.real((np.exp(-phi * np.log(Strike) * 1j) * CF1) / (phi * 1j))
    
    return Output
def NUM_INTEGRATION_HESTON(Function, aLim, bLim, nDiv, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike):
    # nDiv must be even
    Delta  = (bLim - aLim) / nDiv
    
    EveryX = np.linspace(aLim, bLim, nDiv + 1)
    
    EveryY = Function(EveryX, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike)
    
    Integral = Delta/3 * np.sum(EveryY[0:-1:2] + 4*EveryY[1::2] + EveryY[2::2])
    
    return Integral
def P1P2Heston(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike):
    
    NumInt = NUM_INTEGRATION_HESTON(IntFuncHeston, 0.0001, 200, 500, Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike)
    
    PP = 0.5 + (1 / np.pi) * NumInt
    
    return PP
    
def CallHestonCForm(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, Strike):
    
    P1 = P1P2Heston(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, 1, Strike)
    P2 = P1P2Heston(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, 2, Strike)
    
    CallValue = Spot * np.exp(-div * Maturity) * P1 - Strike * np.exp(-rate * Maturity) * P2
    
    return CallValue
Spot = 100
Maturity = 1
kappa = 1.5
rho = -0.4
volvol = 0.6
theta = 0.03
var0 = 0.014
rate = 0.02
div = 0.005
P1P2 = 1 # enter 1 for P1 or 2 for P2
Strike = 101

print(P1P2Heston(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, P1P2, Strike))
print(CallHestonCForm(Spot, Maturity, kappa, rho, volvol, theta, var0, rate, div, Strike))